# Original LANDER Baseline

**Purpose:** Reproduce LANDER (fixed radius r=0.015) with exact paper parameters to get ground-truth baseline numbers on our hardware.


In [1]:
# ============================================================
# Cell: Install
# ============================================================
import subprocess, sys, os
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
install("transformers")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HF authenticated.")
except: print("HF auth skipped.")
print("Done.")


HF auth skipped.
Done.


In [13]:
# ============================================================
# Cell: Imports & Config — PAPER-EXACT PARAMETERS
# ============================================================
import os, copy, math, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

SEED = 2023
def setup_seed(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
setup_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ---- PAPER-EXACT CONFIG (Table 1 / Section 5.1 / Algorithm 1) ----
CFG = dict(
    num_classes      = 100,
    num_tasks        = 5,
    classes_per_task = 20,
    num_clients      = 5,           # paper: 5
    beta             = 0.5,         # paper: NIID(0.5)
    com_rounds       = 100,         # paper: 100
    local_epochs     = 2,           # paper: 2
    local_bs         = 128,         # paper: 128
    local_lr         = 0.04,        # paper: 4e-2
    weight_decay     = 5e-4,        # paper: 5e-4
    # Data-free generation — paper exact
    syn_rounds       = 40,          # paper: 40 (I in Algorithm 1)
    g_steps          = 40,          # paper: 40 (g in Algorithm 1)
    warmup           = 10,          # paper: 10
    lr_g             = 2e-3,        # paper: 2e-3
    syn_batch_size   = 256,         # paper: 256
    syn_total        = 10000,       # paper: 10000
    # Loss weights — paper Section 4.4
    bn_weight        = 1.0,         # lambda_bn = 1.0
    oh_weight        = 0.5,         # lambda_oh = 0.5
    adv_weight       = 1.0,         # adversarial weight
    ltc_weight       = 5.0,         # lambda_ltc = 5
    # LANDER specific — paper Section 5.3
    lander_r         = 0.015,       # fixed radius r
    # Distillation
    kd_temperature   = 2.0,         # tau = 2
    alpha_cur_base   = 0.2,         # paper: alpha_cur
    alpha_pre_base   = 0.4,         # paper: alpha_pre
    # Architecture
    lte_dim          = 512,         # CLIP embedding dim
    feature_dim      = 512,         # ResNet-18 feature dim
    nz               = 256,         # noisy layer output dim
    seed             = SEED,
    # STOP AFTER THIS MANY TASKS
    stop_after_task  = 5,           
)
print("LANDER Baseline — Paper-Exact Parameters")
print("  Rounds/task:", CFG['com_rounds'])
print("  Clients:", CFG['num_clients'])
print("  Syn rounds:", CFG['syn_rounds'], "x", CFG['g_steps'], "steps =", CFG['syn_rounds']*CFG['g_steps'], "gen iters/task")
print("  Syn batch:", CFG['syn_batch_size'], "=> ~%d images/task" % ((CFG['syn_rounds']-CFG['warmup'])*CFG['syn_batch_size']))
print("  Will stop after Task", CFG['stop_after_task']-1)


Device: cuda
LANDER Baseline — Paper-Exact Parameters
  Rounds/task: 100
  Clients: 5
  Syn rounds: 40 x 40 steps = 1600 gen iters/task
  Syn batch: 256 => ~7680 images/task
  Will stop after Task 4


In [14]:
# ============================================================
# Cell: CIFAR-100 Class Names + CLIP Embeddings
# ============================================================
CIFAR100_CLASSES = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle',
    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel',
    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock',
    'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur',
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster',
    'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
    'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
    'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
    'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine',
    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea',
    'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider',
    'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank',
    'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip',
    'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm'
]

def compute_clip_embeddings(names):
    try:
        from transformers import CLIPTokenizer, CLIPTextModel
        print("Loading CLIP ViT-B/32...")
        tok = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
        model = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
        model.eval()
        prompts = ["a photo of a " + n.replace('_', ' ') for n in names]
        embs = []
        with torch.no_grad():
            for p in prompts:
                t = tok(p, return_tensors="pt", padding=True, truncation=True)
                embs.append(model(**t).pooler_output.squeeze(0))
        return F.normalize(torch.stack(embs), dim=-1)
    except Exception as e:
        print("CLIP failed:", e)
        torch.manual_seed(42)
        return F.normalize(torch.randn(len(names), 512), dim=-1)

LTE = compute_clip_embeddings(CIFAR100_CLASSES)
print("LTE:", LTE.shape)


Loading CLIP ViT-B/32...


The following layers were not sharded: text_model.encoder.layers.*.self_attn.v_proj.bias, text_model.encoder.layers.*.mlp.fc1.bias, text_model.embeddings.token_embedding.weight, text_model.encoder.layers.*.self_attn.out_proj.weight, text_model.encoder.layers.*.layer_norm1.weight, text_model.encoder.layers.*.mlp.fc1.weight, text_model.encoder.layers.*.self_attn.q_proj.weight, text_model.encoder.layers.*.self_attn.q_proj.bias, text_model.encoder.layers.*.mlp.fc2.weight, text_model.encoder.layers.*.mlp.fc2.bias, text_model.encoder.layers.*.layer_norm1.bias, text_model.encoder.layers.*.self_attn.k_proj.bias, text_model.encoder.layers.*.layer_norm2.bias, text_model.encoder.layers.*.self_attn.out_proj.bias, text_model.encoder.layers.*.self_attn.v_proj.weight, text_model.final_layer_norm.weight, text_model.final_layer_norm.bias, text_model.encoder.layers.*.layer_norm2.weight, text_model.embeddings.position_embedding.weight, text_model.encoder.layers.*.self_attn.k_proj.weight


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

LTE: torch.Size([100, 512])


In [15]:
# ============================================================
# Cell: Data Loading
# ============================================================
CIFAR_MEAN, CIFAR_STD = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
test_tf = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_data = datasets.CIFAR100('./data', train=True, download=True, transform=train_tf)
test_data  = datasets.CIFAR100('./data', train=False, download=True, transform=test_tf)
train_labels = np.array(train_data.targets)
test_labels  = np.array(test_data.targets)

np.random.seed(SEED)
class_order = np.random.permutation(100).tolist()
label_map = {orig: new for new, orig in enumerate(class_order)}
train_labels_mapped = np.array([label_map[l] for l in train_labels])
test_labels_mapped  = np.array([label_map[l] for l in test_labels])
LTE_ORDERED = LTE[class_order].to(DEVICE)

print("Train:", len(train_labels), "Test:", len(test_labels))

def partition_data_dirichlet(labels, beta, n_parties, task_classes):
    mask = np.isin(labels, task_classes)
    idx = np.where(mask)[0]; tl = labels[idx]
    if beta == 0:
        np.random.shuffle(idx)
        sp = np.array_split(idx, n_parties)
        return {i: sp[i].tolist() for i in range(n_parties)}
    ms = 0
    while ms < 2:
        ib = [[] for _ in range(n_parties)]
        for k in task_classes:
            ik = idx[tl == k]; np.random.shuffle(ik)
            pr = np.random.dirichlet(np.repeat(beta, n_parties))
            pr = (np.cumsum(pr)*len(ik)).astype(int)[:-1]
            for j, s in enumerate(np.split(ik, pr)): ib[j].extend(s.tolist())
        ms = min(len(x) for x in ib)
    for j in range(n_parties): np.random.shuffle(ib[j])
    return {j: ib[j] for j in range(n_parties)}

class IndexedDataset(Dataset):
    def __init__(self, base, indices, labels):
        self.base, self.indices, self.labels = base, indices, labels
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        img, _ = self.base[i]
        return idx, img, torch.tensor(self.labels[i], dtype=torch.long)

class SimpleImageDataset(Dataset):
    def __init__(self, images):
        self.images = images
        self.normalize = transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
        self.augment = transforms.Compose([
            transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = self.augment(self.images[idx])
        return self.normalize(img)

print("Data ready.")


Train: 50000 Test: 10000
Data ready.


In [16]:
# ============================================================
# Cell: Model (ResNet-18)
# ============================================================
from torchvision.models import resnet18 as tv_resnet18

class IncrementalResNet(nn.Module):
    def __init__(self, feat_dim=512, lte_dim=512):
        super().__init__()
        bb = tv_resnet18(weights=None)
        bb.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        bb.maxpool = nn.Identity()
        self.feature_dim = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.mapping = nn.Linear(self.feature_dim, lte_dim)
        self.fc = None

    def update_fc(self, n):
        new_fc = nn.Linear(self.feature_dim, n)
        nn.init.kaiming_uniform_(new_fc.weight, nonlinearity='linear')
        nn.init.constant_(new_fc.bias, 0)
        if self.fc is not None:
            old = self.fc.out_features
            new_fc.weight.data[:old] = self.fc.weight.data.cpu().clone()
            new_fc.bias.data[:old]   = self.fc.bias.data.cpu().clone()
        self.fc = new_fc.to(next(self.backbone.parameters()).device)

    def forward(self, x):
        f = self.backbone(x)
        return {'logits': self.fc(f) if self.fc else None, 'features': f, 'att': self.mapping(f)}

    def copy(self): return copy.deepcopy(self)
    def freeze(self):
        for p in self.parameters(): p.requires_grad = False
        self.eval(); return self

_m = IncrementalResNet().to(DEVICE); _m.update_fc(20)
_o = _m(torch.randn(2,3,32,32,device=DEVICE))
print("Model OK:", _o['logits'].shape, _o['att'].shape)
del _m, _o; torch.cuda.empty_cache()


Model OK: torch.Size([2, 20]) torch.Size([2, 512])


In [17]:
# ============================================================
# Cell: Generator (paper-exact NoisyLayerGenerator)
# ============================================================
class NoisyLayerGenerator(nn.Module):
    def __init__(self, label_emb, ngf=64, img_size=32, nc=3,
                 nl=10, le_emb_size=256, le_size=512, sbz=256):
        super().__init__()
        self.label_emb = nn.Parameter(label_emb, requires_grad=False)
        self.init_size = img_size // 4
        self.nl, self.nle = nl, int(np.ceil(sbz / nl))
        self.n1 = nn.BatchNorm1d(le_size)
        self.le1 = nn.ModuleList([nn.Linear(le_size, le_emb_size) for _ in range(self.nle)])
        self.l1 = nn.Sequential(nn.Linear(le_emb_size, ngf*2*self.init_size**2))
        self.conv_blocks = nn.Sequential(
            nn.BatchNorm2d(ngf*2), nn.Upsample(scale_factor=2),
            nn.Conv2d(ngf*2,ngf*2,3,1,1,bias=False), nn.BatchNorm2d(ngf*2), nn.LeakyReLU(0.2,True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(ngf*2,ngf,3,1,1,bias=False), nn.BatchNorm2d(ngf), nn.LeakyReLU(0.2,True),
            nn.Conv2d(ngf,nc,3,1,1), nn.Sigmoid())
    def forward(self, targets):
        le = self.n1(self.label_emb[targets])
        v = None
        for i in range(self.nle):
            s,e = i*self.nl, min((i+1)*self.nl, le.shape[0])
            sv = self.le1[i](le[s:e])
            v = sv if v is None else torch.cat((v,sv))
        out = self.l1(v).view(v.shape[0],-1,self.init_size,self.init_size)
        return self.conv_blocks(out)
    def re_init_noisy_layer(self):
        for i in range(self.nle):
            nn.init.normal_(self.le1[i].weight,0,1)
            nn.init.constant_(self.le1[i].bias,0)
print("Generator ready.")


Generator ready.


In [18]:
# ============================================================
# Cell: Losses & Utilities (LANDER only — fixed radius)
# ============================================================
class DeepInversionHook:
    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn); self.r_feature = None
    def hook_fn(self, module, inp, out):
        nch = inp[0].shape[1]
        mean = inp[0].mean([0,2,3])
        var = inp[0].permute(1,0,2,3).contiguous().view([nch,-1]).var(1, unbiased=False)
        self.r_feature = torch.norm(module.running_var.data-var,2)+torch.norm(module.running_mean.data-mean,2)
    def remove(self): self.hook.remove()

def bounding_loss_fixed(features, targets_emb, r):
    return torch.relu(F.mse_loss(features, targets_emb.detach()) - r)

def kd_loss(s, t, T=2.0):
    return F.kl_div(F.log_softmax(s/T,1), F.softmax(t/T,1), reduction='batchmean')*T*T

def fedavg(ws):
    avg = copy.deepcopy(ws[0])
    for k in avg:
        for i in range(1,len(ws)): avg[k] = avg[k]+ws[i][k]
        if avg[k].is_floating_point(): avg[k] = avg[k]/len(ws)
        else: avg[k] = avg[k]//len(ws)
    return avg

print("Losses ready (fixed radius LANDER only).")


Losses ready (fixed radius LANDER only).


In [19]:
# ============================================================
# Cell: Data-Free Generator (LANDER — fixed radius)
# ============================================================
class DataFreeGenerator:
    def __init__(self, teacher, generator, num_classes, label_emb, cfg):
        self.teacher = teacher.eval()
        self.generator = generator.to(DEVICE).train()
        self.num_classes, self.label_emb, self.cfg = num_classes, label_emb, cfg
        self.hooks = [DeepInversionHook(m) for m in teacher.modules() if isinstance(m, nn.BatchNorm2d)]
        self.student = copy.deepcopy(teacher)
        for p in self.student.parameters():
            if p.dim()>=2: p.data.normal_(0,0.01)
            else: p.data.zero_()
        self.mean = torch.tensor([0.5,0.5,0.5], requires_grad=True, device=DEVICE)
        self.std  = torch.tensor([0.2,0.2,0.2], requires_grad=True, device=DEVICE)

    def _make_targets(self, bs):
        s,v = bs//self.num_classes, bs%self.num_classes
        t = torch.randint(self.num_classes,(v,))
        for _ in range(s): t = torch.cat([torch.arange(self.num_classes),t])
        t = t[:bs].to(DEVICE)
        ys = torch.zeros(bs,self.num_classes,device=DEVICE); ys.scatter_(1,t.unsqueeze(1),1.0)
        return t, ys

    def synthesize_round(self, rnd):
        self.teacher.eval(); self.student.eval(); self.generator.train()
        self.generator.re_init_noisy_layer()
        targets, ys = self._make_targets(self.cfg['syn_batch_size'])
        opt = torch.optim.Adam([{'params':self.generator.parameters()},
            {'params':[self.mean],'lr':0.01},{'params':[self.std],'lr':0.01}],
            lr=self.cfg['lr_g'], betas=[0.5,0.999])
        best_inputs, best_cost = None, 1e6
        for _ in range(self.cfg['g_steps']):
            imgs = self.generator(targets)
            imgs_n = (imgs-self.mean[None,:,None,None])/(self.std[None,:,None,None]+1e-6)
            t_out = self.teacher(imgs_n)
            loss_bn = sum(h.r_feature for h in self.hooks if h.r_feature is not None)
            loss_oh = -(ys*F.log_softmax(t_out['logits'],-1)).sum(-1).mean()
            temb = self.label_emb[targets]
            loss_ltc = bounding_loss_fixed(t_out['att'], temb, self.cfg['lander_r'])
            loss_adv = torch.tensor(0.,device=DEVICE)
            if rnd > self.cfg['warmup']:
                s_out = self.student(imgs_n)['logits']
                mask = (s_out.argmax(1)==t_out['logits'].argmax(1)).float()
                loss_adv = -(F.kl_div(F.log_softmax(s_out/2,1),F.softmax(t_out['logits']/2,1),
                    reduction='none').sum(1)*mask).mean()
            loss = self.cfg['bn_weight']*loss_bn+self.cfg['oh_weight']*loss_oh+                   self.cfg['adv_weight']*loss_adv+self.cfg['ltc_weight']*loss_ltc
            with torch.no_grad():
                if loss.item()<best_cost: best_cost=loss.item(); best_inputs=imgs.data.clone()
            opt.zero_grad(); loss.backward(); opt.step()
        if rnd>self.cfg['warmup'] and best_inputs is not None:
            self._train_student(best_inputs)
        return best_inputs

    def _train_student(self, imgs):
        self.student.train(); self.teacher.eval()
        opt = torch.optim.SGD(self.student.parameters(), lr=0.1, momentum=0.9)
        imgs_n = ((imgs-self.mean[None,:,None,None])/(self.std[None,:,None,None]+1e-6)).detach()
        for _ in range(50):  # paper: student_train_step=50
            with torch.no_grad(): t = self.teacher(imgs_n)
            s = self.student(imgs_n)
            loss = kd_loss(s['logits'],t['logits'].detach(),20.)+F.mse_loss(s['att'],t['att'].detach())
            opt.zero_grad(); loss.backward(); opt.step()
        self.student.eval()

    def generate_all(self):
        all_imgs = []
        total = self.cfg['syn_rounds']+self.cfg['warmup']
        for r in range(total):
            imgs = self.synthesize_round(r)
            if r>=self.cfg['warmup'] and imgs is not None: all_imgs.append(imgs.cpu())
        if not all_imgs: return []
        all_imgs = torch.cat(all_imgs)
        if len(all_imgs)>self.cfg['syn_total']:
            all_imgs = all_imgs[torch.randperm(len(all_imgs))[:self.cfg['syn_total']]]
        return list(all_imgs)

    def cleanup(self):
        for h in self.hooks: h.remove()

print("DataFreeGenerator ready (fixed radius, student_train_step=50).")


DataFreeGenerator ready (fixed radius, student_train_step=50).


In [22]:
# ============================================================
# Cell: LANDER Trainer (Paper-Exact)
# ============================================================
class LANDERTrainer:
    def __init__(self, cfg, lte):
        self.cfg, self.lte = cfg, lte
        self.model = IncrementalResNet(cfg['feature_dim'], cfg['lte_dim']).to(DEVICE)
        self.old_model = None
        self.known_classes = 0
        self.total_classes = 0
        self.syn_images = []
        self.results = {'per_task_acc': [], 'task_acc_matrix': []}

    def _local_update_task0(self, model, loader, lr):
        model.train()
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=self.cfg['weight_decay'])
        tl = 0
        for ep in range(self.cfg['local_epochs']):
            for _, imgs, labs in loader:
                imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
                out = model(imgs)
                loss = F.cross_entropy(out['logits'], labs) +                        self.cfg['ltc_weight'] * bounding_loss_fixed(out['att'], self.lte[labs], self.cfg['lander_r'])
                opt.zero_grad(); loss.backward(); opt.step()
                if ep==0: tl += loss.item()
        return model.state_dict(), tl

    def _local_finetune(self, old_model, model, loader, syn_loader, lr):
        model.train(); old_model.eval()
        a = np.log2(self.total_classes/2+1)
        b = np.sqrt(self.known_classes/self.total_classes)
        cw = self.cfg['alpha_cur_base']*(1+1/a)/b
        pw = self.cfg['alpha_pre_base']*a*b
        opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=self.cfg['weight_decay'])
        syn_iter = iter(syn_loader)
        tl = 0
        for ep in range(self.cfg['local_epochs']):
            for _, imgs, labs in loader:
                imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
                try: syn = next(syn_iter).to(DEVICE)
                except StopIteration: syn_iter=iter(syn_loader); syn=next(syn_iter).to(DEVICE)
                c = model(imgs)
                loss_ce = F.cross_entropy(c['logits'][:,self.known_classes:], labs-self.known_classes)
                loss_b = bounding_loss_fixed(c['att'], self.lte[labs], self.cfg['lander_r'])
                s = model(syn)
                with torch.no_grad(): t = old_model(syn)
                loss_kd = kd_loss(s['logits'][:,:self.known_classes], t['logits'].detach(), self.cfg['kd_temperature'])
                loss_f = F.mse_loss(s['att'], t['att'].detach())
                loss = cw*(loss_ce+self.cfg['ltc_weight']*loss_b)+pw*(loss_kd+loss_f)
                opt.zero_grad(); loss.backward(); opt.step()
                if ep==0: tl+=loss.item()
        return model.state_dict(), tl

    def _fl_train(self, tid, task_classes):
        ug = partition_data_dirichlet(train_labels_mapped, self.cfg['beta'], self.cfg['num_clients'], task_classes)
        sl = None
        if tid>0 and self.syn_images:
            sl = DataLoader(SimpleImageDataset(self.syn_images), self.cfg['local_bs'], shuffle=True, drop_last=True)
        for rnd in range(self.cfg['com_rounds']):
            lr = self.cfg['local_lr']*0.5*(1+np.cos(np.pi*rnd/self.cfg['com_rounds']))
            ws = []
            for cid in range(self.cfg['num_clients']):
                ds = IndexedDataset(train_data, ug[cid], train_labels_mapped)
                dl = DataLoader(ds, self.cfg['local_bs'], shuffle=True,
                                drop_last=len(ug[cid])>self.cfg['local_bs'])
                cm = copy.deepcopy(self.model)
                if tid==0: w,_=self._local_update_task0(cm,dl,lr)
                else:      w,_=self._local_finetune(self.old_model,cm,dl,sl,lr)
                ws.append(w); del cm; torch.cuda.empty_cache()
            self.model.load_state_dict(fedavg(ws))
            if (rnd+1)%10==0 or rnd==self.cfg['com_rounds']-1:
                acc = self._evaluate(list(range(self.total_classes)))
                print("  [LANDER] Task %d Round %d/%d => %.2f%%" % (tid,rnd+1,self.cfg['com_rounds'],acc))

    def _gen_synthetic(self):
        print("  [LANDER] Generating synthetic data (%d classes, paper-exact)..." % self.total_classes)
        gen = NoisyLayerGenerator(self.lte[:self.total_classes],
            sbz=self.cfg['syn_batch_size'], le_emb_size=self.cfg['nz'], le_size=self.cfg['lte_dim']).to(DEVICE)
        dfg = DataFreeGenerator(copy.deepcopy(self.model).eval(), gen, self.total_classes, self.lte, self.cfg)
        self.syn_images = dfg.generate_all(); dfg.cleanup()
        print("  => %d images generated." % len(self.syn_images))

    def _evaluate(self, cls_range):
        self.model.eval()
        ds = IndexedDataset(test_data, np.where(np.isin(test_labels_mapped,cls_range))[0], test_labels_mapped)
        dl = DataLoader(ds, 256, shuffle=False)
        c=t=0
        with torch.no_grad():
            for _,imgs,labs in dl:
                imgs,labs=imgs.to(DEVICE),labs.to(DEVICE)
                c+=(self.model(imgs)['logits'][:,:max(cls_range)+1].argmax(1)==labs).sum().item()
                t+=labs.size(0)
        self.model.train()
        return 100.*c/t if t else 0.

    def run(self):
        print("\n"+"="*60)
        print("  LANDER Baseline (Paper-Exact Parameters)")
        print("="*60)
        setup_seed(self.cfg['seed'])
        for tid in range(self.cfg['stop_after_task']):
            self.total_classes = self.known_classes + self.cfg['classes_per_task']
            self.model.update_fc(self.total_classes); self.model.to(DEVICE)
            print("\nTask %d: classes %d-%d" % (tid, self.known_classes, self.total_classes-1))
            if tid>0: self._gen_synthetic()
            self._fl_train(tid, list(range(self.known_classes, self.total_classes)))
            acc = self._evaluate(list(range(self.total_classes)))
            self.results['per_task_acc'].append(acc)
            taccs = []
            for t2 in range(tid+1):
                tc = list(range(t2*self.cfg['classes_per_task'], (t2+1)*self.cfg['classes_per_task']))
                taccs.append(self._evaluate(tc))
            self.results['task_acc_matrix'].append(taccs)
            print("  Task %d DONE => Overall: %.2f%%, Per-task: %s" % (tid, acc, ['%.1f'%a for a in taccs]))
            self.old_model = self.model.copy().freeze()
            self.known_classes = self.total_classes
        return self.results

print("LANDERTrainer ready.")


LANDERTrainer ready.


In [23]:
# ============================================================
# Cell: RUN
# ============================================================
print("="*60)
print("  LANDER Baseline — Paper-Exact")
print("  100 rounds/task, 5 clients, 10k syn images, r=0.015")
print("="*60)

t0 = time.time()
trainer = LANDERTrainer(CFG, LTE_ORDERED)
results = trainer.run()
elapsed = time.time() - t0
print("\nLANDER finished in %.1f minutes (%.1f hours)." % (elapsed/60, elapsed/3600))
del trainer; torch.cuda.empty_cache()


  LANDER Baseline — Paper-Exact
  100 rounds/task, 5 clients, 10k syn images, r=0.015

  LANDER Baseline (Paper-Exact Parameters)

Task 0: classes 0-19
  [LANDER] Task 0 Round 10/100 => 54.00%
  [LANDER] Task 0 Round 20/100 => 66.35%
  [LANDER] Task 0 Round 30/100 => 71.15%
  [LANDER] Task 0 Round 40/100 => 75.65%
  [LANDER] Task 0 Round 50/100 => 77.05%
  [LANDER] Task 0 Round 60/100 => 78.50%
  [LANDER] Task 0 Round 70/100 => 79.20%
  [LANDER] Task 0 Round 80/100 => 79.70%
  [LANDER] Task 0 Round 90/100 => 79.75%
  [LANDER] Task 0 Round 100/100 => 79.60%
  Task 0 DONE => Overall: 79.60%, Per-task: ['79.6']

Task 1: classes 20-39
  [LANDER] Generating synthetic data (40 classes, paper-exact)...
  => 10000 images generated.
  [LANDER] Task 1 Round 10/100 => 51.92%
  [LANDER] Task 1 Round 20/100 => 57.25%
  [LANDER] Task 1 Round 30/100 => 60.70%
  [LANDER] Task 1 Round 40/100 => 61.95%
  [LANDER] Task 1 Round 50/100 => 62.40%
  [LANDER] Task 1 Round 60/100 => 64.47%
  [LANDER] Task 1 Ro

In [11]:
# ============================================================
# Cell: Results
# ============================================================
print("\n" + "="*60)
print("  LANDER Baseline Results (Paper-Exact Parameters)")
print("="*60)
for i, acc in enumerate(results['per_task_acc']):
    print("  After Task %d: %.2f%%" % (i, acc))
    print("    Per-task: %s" % ['%.1f%%'%a for a in results['task_acc_matrix'][i]])

print("\nComparison with paper (NIID beta=0.5, T=5):")
paper_vals = [76.0, 62.0]  # approximate from Figure 4
for i in range(len(results['per_task_acc'])):
    ours = results['per_task_acc'][i]
    paper = paper_vals[i] if i < len(paper_vals) else None
    if paper:
        print("  Task %d: Ours=%.2f%% Paper~%.1f%% Delta=%+.2f%%" % (i, ours, paper, ours-paper))
    else:
        print("  Task %d: Ours=%.2f%%" % (i, ours))

print("\nThese are ground-truth LANDER numbers on YOUR hardware.")
print("Compare directly with SA-LANDER Task 0 and Task 1 results.")



  LANDER Baseline Results (Paper-Exact Parameters)
  After Task 0: 79.60%
    Per-task: ['79.6%']
  After Task 1: 65.45%
    Per-task: ['78.5%', '64.2%']

Comparison with paper (NIID beta=0.5, T=5):
  Task 0: Ours=79.60% Paper~76.0% Delta=+3.60%
  Task 1: Ours=65.45% Paper~62.0% Delta=+3.45%

These are ground-truth LANDER numbers on YOUR hardware.
Compare directly with SA-LANDER Task 0 and Task 1 results.
